# Schema comparison: Keerat vs. Jun vs. Andrei

Prepared for the Wednesday 9/30 ERD-merge meeting.

- Mine: `keerat/02_normalization.ipynb`
- Jun's: `jun` branch, `house/`. The schema on `main` is the **same file** as Jun's.
- Andrei's: `andrei` branch, `own_implementation/`

## 1. Big picture

| | Keerat | Jun | Andrei |
|---|---|---|---|
| Tables | 12 | 21 | 17 |
| Central table | `filing` | `unit_record` | `UnitReport` |
| Has it been loaded / run? | Built in pandas; PK, FK and lossless checks pass. Not in MySQL yet. | Loaded into MySQL 8, validated | "has not run" (per commit message) |
| Source snapshot | 8/30 (549,899 rows) | 9/13 (550,201 rows) | 9/27 (per his notes) |

**All three agree on the core design:**

- One row = one filing about one unit in one year, with key `unique_id`.
- There is **no** property/building table, because block-level facts conflict ~70–85% of the time.
- History JSON becomes a 1-to-many child table.
- The five utility columns become a utility table plus a junction table (many-to-many).
- `point → neighborhood, district` becomes a `location_point` table.
- `submission_year → case_type_name` becomes a filing-year table.
- Bedrooms, bathrooms, rent and sq ft get lookup tables that keep the raw label next to the cleaned value.

So the disagreement is about **how much extra detail** to include, not about the normalization itself.

## 2. Where the designs differ

| Topic | Keerat | Jun | Andrei | Decision needed |
|---|---|---|---|---|
| Export metadata (`data_as_of`, `data_loaded_at`) | dropped | `extract_batch` table | `ExtractBatch` table | Keep a table for it, or drop? |
| Data-quality tracking | invalid values set to NULL, counted in the notebook | `quality_flag` + `record_quality_flag` (keeps every rejected value) | `QualityIssue` | Keep? It's the strongest "we never silently drop data" evidence. |
| Duplicate look-alike rows (~62k) | not modeled | `duplicate_group` table | not modeled | Do we need it for our queries? |
| Blocks | `block_num`, `block_address` stay as columns | `assessor_block` + `street_block` tables | `AssessorBlock` + `BlockAddress` | Tables that only hold a list of IDs add FKs but little meaning. |
| Supervisor district | column on `location_point` | own table | own table | Minor. |
| History type (Occupied/Vacant) | text column | `date_range_type` lookup | text column | Minor. |
| History key | `(unique_id, seq_no)` | `history_id` + unique `(unique_id, seq_no)` | `history_id` | Either is fine. |
| Year-unknown answers | `occupancy_year_unknown` text column | `date_unknown_reason` lookup | `date_unknown_text` column | Minor. |
| Utility vocabulary | 10 utilities + `category` | 16 utilities + checkbox/text flags | ~7 + source flags | Fine either way. |

## 3. Something else to consider

**`occupancy_or_vacancy_date → occupancy_or_vacancy_date_year` is a transitive dependency.**
On all 446,769 rows that have both, the year column equals `YEAR(date)` (0 exceptions).
Jun's `unit_record` and Andrei's `UnitReport` both store the date **and** the year, so that's technically a 3NF violation.

- **Fix:** keep the year only for the 33,393 rows that have a year but no full date (my `occupancy_year_only`), and get it with `YEAR(occupancy_date)` otherwise.
- Small change, but it's exactly the kind of thing the professor may ask about.

## 4. Other things to raise

1. **Pick one CSV snapshot.** Our three files have different row counts. All slide numbers should come from one file.
2. **The ERD on `main` has lines attached to the wrong columns.** For example, `bathroom_type` points at `unit_record.unique_id`, `sqft_band` at `batch_id`, `rent_band` at `submission_year`, and `duplicate_group` at `point_id`. We should regenerate it from the final DDL (e.g. MySQL Workbench → Database → Reverse Engineer), as Andrei's notes suggest.
3. **The `.csv.gz` on `main` isn't data.** It's a 1.7 KB text file. Either remove it or add a README line explaining where to download the CSV.
4. **21 tables is a lot to explain in a mid-presentation.** Option: present the 12 "core" tables and mention the audit tables (quality, duplicates, batch) as extras.
5. **Rent midpoints.** Jun's queries average band midpoints. Andrei's notes say to label that as an assumption. We should agree on wording for the slides.
6. **Who presents what.** Each of us needs 2 basic + 2 advanced queries on the **final** schema, run in MySQL.

## 5. Suggestion

Use **Jun's schema as the base**, since it's the only one proven to load into MySQL. Then:

- apply the date → year fix (section 3)
- regenerate the ERD from the final DDL
- decide whether the block and supervisor-district tables are worth keeping
